In [1]:
import numpy as np
import pandas as pd
import polars as pl
import pickle 
from scipy.sparse import hstack
import warnings
import regex as re
warnings.filterwarnings("ignore")

In [2]:
#!pip -q install /kaggle/input/pytorchtabnet/pytorch_tabnet-4.1.0-py3-none-any.whl

In [3]:
#from pytorch_tabnet.tab_model import TabNetRegressor
#import torch

In [4]:
vers = 86
data = "/kaggle/input/um-game-playing-strength-of-mcts-variants/test.csv"

filepath = f"/kaggle/input/um-v{vers}/v{vers}/" # Correct LGB
filepath2 = f"/kaggle/input/um-v{77}/v{77}/" # best CAT
filepath4 = f"/kaggle/input/um-v{84}/v{84}/"
filepath3 = f"/kaggle/input/um-v{72}/v{72}/" # best LGB
#filepath4 = f"/kaggle/input/um-v{81}/v{81}/" # gkf by agent11 and agent21
#filepath5 = f"/kaggle/input/um-v{74}/V{74}/" # best CV CAT

target = "utility_agent1"

# Features for models
#features = pickle.load(open(filepath + "feats.pkl", "rb"))

# Ordinal Encoder for Object Columns
objs_enc = pickle.load(open(filepath2 + "objs_enc.pkl", "rb"))

# Ordinal Encoder for Agent Columns
split_enc = pickle.load(open(filepath2 + "split_enc.pkl", "rb"))

board_enc = pickle.load(open(filepath2 + "boards_enc.pkl", "rb"))

# Char Vectorizer for EnglishRules
vect1 = pickle.load(open(filepath2 + "vect1.pkl", "rb"))

# Word Vectorizer for English Rules
vect2 = pickle.load(open(filepath2 + "vect2.pkl", "rb"))

#tfidf_vect = pickle.load(open(filepath + "tfidf_vect.pkl", "rb"))

#same_games_dict = pickle.load(open("/kaggle/input/same-games-dict/same_games_dict.pkl", "rb"))
splits = 10

In [5]:
models = []
models2 = []
models3 = []


models2.extend([pickle.load(open(filepath2 + f"cat{idx}.pkl", "rb")) for idx in range(splits)])
models.extend([pickle.load(open(filepath + f"lgbAgent1{idx}.pkl", "rb")) for idx in range(splits)])
#models3.extend([pickle.load(open(filepath3 + f"cat{idx}.pkl", "rb")) for idx in range(splits)])
#models3.extend([pickle.load(open(filepath3 + f"lgbAgent1{idx}.pkl", "rb")) for idx in range(splits)])
#models4.extend([pickle.load(open(filepath4 + f"cat{idx}.pkl", "rb")) for idx in range(4)])
#models5.extend([pickle.load(open(filepath4 + f"cat2{idx}.pkl", "rb")) for idx in range(4)])
#models6.extend([pickle.load(open(filepath5 + f"cat{idx}.pkl", "rb")) for idx in range(splits)])


In [6]:
#features1 = models3[0].feature_name_
# For LGB MODEL
features = models[0].feature_name_

# for CAT V2 MODEL
features1 = pickle.load(open(filepath4 + "cols.pkl", "rb"))
features1 = [i for i in features1] 
#features1.extend(["lgb_preds"])
#features = pickle.load(open(filepath4 + "features.pkl", "rb"))

In [7]:
obj = ['Id','GameRulesetName', 'agent1', 'agent2', 'EnglishRules', 'LudRules']
obj += ['Behaviour',
 'StateRepetition',
 'Duration',
 'Complexity',
 'BoardCoverage',
 'GameOutcome',
 'StateEvaluation',
 'Clarity',
 'Decisiveness',
 'Drama',
 'MoveEvaluation',
 'StateEvaluationDifference',
 'BoardSitesOccupied',
 'BranchingFactor',
 'DecisionFactor',
 'MoveDistance',
 'PieceNumber',
 'ScoreDifference']

In [8]:
useless = ['Properties',
 'Format',
 'Time',
 'Discrete',
 'Realtime',
 'Turns',
 'Alternating',
 'Simultaneous',
 'HiddenInformation',
 'Match',
 'AsymmetricRules',
 'AsymmetricPlayRules',
 'AsymmetricEndRules',
 'AsymmetricSetup',
 'Players',
 'NumPlayers',
 'Simulation',
 'Solitaire',
 'TwoPlayer',
 'Multiplayer',
 'Coalition',
 'Puzzle',
 'DeductionPuzzle',
 'PlanningPuzzle',
 'Equipment',
 'Container',
 'Board',
 'PrismShape',
 'ParallelogramShape',
 'RectanglePyramidalShape',
 'TargetShape',
 'BrickTiling',
 'CelticTiling',
 'QuadHexTiling',
 'Hints',
 'PlayableSites',
 'Component',
 'DiceD3',
 'BiasedDice',
 'Card',
 'Domino',
 'Rules',
 'SituationalTurnKo',
 'SituationalSuperko',
 'InitialAmount',
 'InitialPot',
 'Play', 
 'BetDecision',
 'BetDecisionFrequency',
 'VoteDecisionFrequency',
 'ChooseTrumpSuitDecision',
 'ChooseTrumpSuitDecisionFrequency',
 'LeapDecisionToFriend',
 'LeapDecisionToFriendFrequency',
 'HopDecisionEnemyToFriend',
 'HopDecisionEnemyToFriendFrequency',
 'HopDecisionFriendToFriend',
 'FromToDecisionWithinBoard',
 'FromToDecisionBetweenContainers',
 'BetEffect',
 'BetEffectFrequency',
 'VoteEffectFrequency',
 'SwapPlayersEffectFrequency',
 'TakeControl',
 'TakeControlFrequency',
 'PassEffectFrequency',
 'SetCost',
 'SetCostFrequency',
 'SetPhase',
 'SetPhaseFrequency',
 'SetTrumpSuit',
 'SetTrumpSuitFrequency',
 'StepEffectFrequency',
 'SlideEffectFrequency',
 'LeapEffectFrequency',
 'HopEffectFrequency',
 'FromToEffectFrequency',
 'SwapPiecesEffect',
 'SwapPiecesEffectFrequency',
 'ShootEffect',
 'ShootEffectFrequency',
 'MaxCapture',
 'OffDiagonalDirection',
 'Information',
 'HidePieceType',
 'HidePieceOwner',
 'HidePieceCount',
 'HidePieceRotation',
 'HidePieceValue',
 'HidePieceState',
 'InvisiblePiece',
 'End',
 'LineDrawFrequency',
 'ConnectionDraw',
 'ConnectionDrawFrequency',
 'GroupLossFrequency',
 'GroupDrawFrequency',
 'LoopLossFrequency',
 'LoopDraw',
 'LoopDrawFrequency',
 'PatternLoss',
 'PatternLossFrequency',
 'PatternDraw',
 'PatternDrawFrequency',
 'PathExtentEndFrequency',
 'PathExtentWinFrequency',
 'PathExtentLossFrequency',
 'PathExtentDraw',
 'PathExtentDrawFrequency',
 'TerritoryLoss',
 'TerritoryLossFrequency',
 'TerritoryDraw',
 'TerritoryDrawFrequency',
 'CheckmateLoss',
 'CheckmateLossFrequency',
 'CheckmateDraw',
 'CheckmateDrawFrequency',
 'NoTargetPieceLoss',
 'NoTargetPieceLossFrequency',
 'NoTargetPieceDraw',
 'NoTargetPieceDrawFrequency',
 'NoOwnPiecesDraw',
 'NoOwnPiecesDrawFrequency',
 'FillLoss',
 'FillLossFrequency',
 'FillDraw',
 'FillDrawFrequency',
 'ScoringDrawFrequency',
 'NoProgressWin',
 'NoProgressWinFrequency',
 'NoProgressLoss',
 'NoProgressLossFrequency',
 'SolvedEnd',
 'Behaviour',
 'StateRepetition',
 'PositionalRepetition',
 'SituationalRepetition',
 'Duration',
 'Complexity',
 'BoardCoverage',
 'GameOutcome',
 'StateEvaluation',
 'Clarity',
 'Narrowness',
 'Variance',
 'Decisiveness',
 'DecisivenessMoves',
 'DecisivenessThreshold',
 'LeadChange',
 'Stability',
 'Drama',
 'DramaAverage',
 'DramaMedian',
 'DramaMaximum',
 'DramaMinimum',
 'DramaVariance',
 'DramaChangeAverage',
 'DramaChangeSign',
 'DramaChangeLineBestFit',
 'DramaChangeNumTimes',
 'DramaMaxIncrease',
 'DramaMaxDecrease',
 'MoveEvaluation',
 'MoveEvaluationAverage',
 'MoveEvaluationMedian',
 'MoveEvaluationMaximum',
 'MoveEvaluationMinimum',
 'MoveEvaluationVariance',
 'MoveEvaluationChangeAverage',
 'MoveEvaluationChangeSign',
 'MoveEvaluationChangeLineBestFit',
 'MoveEvaluationChangeNumTimes',
 'MoveEvaluationMaxIncrease',
 'MoveEvaluationMaxDecrease',
 'StateEvaluationDifference',
 'StateEvaluationDifferenceAverage',
 'StateEvaluationDifferenceMedian',
 'StateEvaluationDifferenceMaximum',
 'StateEvaluationDifferenceMinimum',
 'StateEvaluationDifferenceVariance',
 'StateEvaluationDifferenceChangeAverage',
 'StateEvaluationDifferenceChangeSign',
 'StateEvaluationDifferenceChangeLineBestFit',
 'StateEvaluationDifferenceChangeNumTimes',
 'StateEvaluationDifferenceMaxIncrease',
 'StateEvaluationDifferenceMaxDecrease',
 'BoardSitesOccupied',
 'BoardSitesOccupiedMinimum',
 'BranchingFactor',
 'BranchingFactorMinimum',
 'DecisionFactor',
 'DecisionFactorMinimum',
 'MoveDistance',
 'MoveDistanceMinimum',
 'PieceNumber',
 'PieceNumberMinimum',
 'ScoreDifference',
 'ScoreDifferenceMinimum',
 'ScoreDifferenceChangeNumTimes',
 'Roots',
 'Cosine',
 'Sine',
 'Tangent',
 'Exponential',
 'Logarithm',
 'ExclusiveDisjunction',
 'Float',
 'HandComponent',
 'SetHidden',
 'SetInvisible',
 'SetHiddenCount',
 'SetHiddenRotation',
 'SetHiddenState',
 'SetHiddenValue',
 'SetHiddenWhat',
 'SetHiddenWho']

In [9]:
obj += useless

In [10]:
def get_feature(df, op, col):
    if op == "+":
        sp = col.split(op)
        first = sp[0]
        second = sp[1]
        col = df[first] + df[second]
    elif op == "-":
        sp = col.split(op)
        first = sp[0]
        second = sp[1]
        col = df[first] - df[second]
    elif op == "*":
        sp = col.split(op)
        first = sp[0]
        second = sp[1]
        col = df[first] * df[second]
    elif op == "/":
        sp = col.split(op)
        first = sp[0]
        second = sp[1]
        col = df[first] / df[second]
    return col
    

def get_brute_features(df, cols):
    dummy = pd.DataFrame()
    for col in cols:
        if re.search("\+", col):
            dummy[col] = get_feature(df, "+", col)
        elif re.search("\-", col):
            dummy[col] = get_feature(df, "-", col)
        elif re.search("\*", col):
            dummy[col] = get_feature(df, "*", col)
        elif re.search("\/", col):
            dummy[col] = get_feature(df, "/", col)
    return dummy

In [11]:
def find_pattern(df, pattern):
    li = []
    for row in df["GameRulesetName"]:
        if re.search(pattern, row, re.IGNORECASE):
            li.append(1)
        else:
            li.append(0)
    return li

def get_pattern_df(df, patterns):
    pattern_df = pd.DataFrame()
    
    for pattern in patterns:
        li = find_pattern(df, pattern)
        pattern_df[pattern] = li 
    
    return pattern_df

patterns = ["observe", "suggest", "describe", "diagonal", "join", "\dx\d", "scholar", "jump"]

In [12]:
def get_Boards_feat(df):
    li = []

    for idx,row in enumerate(df["LudRules"]):
        if re.search("\(board \([a-z ]+", row, re.IGNORECASE):
            span = re.search("\(board \([a-z ]+", row, re.IGNORECASE).span()
            #print(span[0], span[1])
            li.append(row[span[0]+8:span[1]].strip())

        elif re.search("\(boardless [a-z ]+", row, re.IGNORECASE):
            span = re.search("\(boardless [a-z ]+", row, re.IGNORECASE).span()
            #print(span[0], span[1])
            li.append(row[span[0]+8:span[1]].strip())
        elif re.search("mancalaboard", row, re.IGNORECASE):
            li.append("mancalaBoard")
        elif re.search("surakartaBoard", row, re.IGNORECASE):
            li.append("surakartaBoard")
        else:
            li.append("")
    return pd.Series(li)


In [13]:
def get_agg_features(df, grps, aggs, cols, di):
    #li = []
    dummy = pd.DataFrame()
    #di = {}
    
    for col in cols:
        for agg_ in aggs:
            
            ag = grps.map(di[col + "_" + agg_])
            #di[col +"_" + agg_] = df.groupby(grps)[col].agg(agg_).to_dict()
            dummy["agg" + col + agg_ + "_diff"] = df[col] - ag
            #li.append("agg" + col + agg_ + "_diff")
            
            dummy["agg" + col + agg_ + "_diff_%"] = dummy["agg" +col + agg_ + "_diff"] / ag
            #li.append("agg" + col + agg_+ "_diff_%")
            
            
            dummy["agg" + col + agg_ + "_diff_20"] = df[col] - (ag * .2)
            #li.append("agg" + col + agg_ + "_diff_20")
            
            dummy["agg" + col + agg_ + "_diff_20_%"] = df[col] - (ag * .2) / ((ag * .2))
            #li.append("agg" + col + agg_ + "_diff_20_%")
            
            
            dummy["agg" + col + agg_ + "_dif_40"] = df[col] - (ag * .4)
            #li.append("agg" + col + agg_ + "_dif_40")
            
            dummy["agg" + col + agg_ + "_dif_40_%"] = df[col] - (ag * .4) / ((ag * .4))
            #li.append("agg" + col + agg_ + "_dif_40_%")
            
            
            dummy["agg" + col + agg_ + "_diff_60"] = df[col] - (ag * .6)
            #li.append("agg" + col + agg_ + "_diff_60")
            
            dummy["agg" + col + agg_ + "_diff_60_%"] = df[col] - (ag * .6) / ((ag * .6))
            #li.append("agg" + col + agg_ + "_diff_60_%")
            
            
            dummy["agg" + col + agg_ + "_diff_80"] = df[col] - (ag * .8) 
            #li.append("agg" + col + agg_ + "_diff_80")

            dummy["agg" + col + agg_ + "_diff_80_%"] = df[col] - (ag * .8) / ((ag * .8))
            #li.append("agg" + col + agg_ + "_diff_80_%")
            
    return dummy#, li# di

In [14]:
aggs = ["min", "max", "mean", "median"]
# Min Imp 146
cols = ["AdvantageP1", 
        "GameTreeComplexity", 
        "DurationTurnsStdDev", 
        "Balance", 
        "OutcomeUniformity", 
        "BoardSitesOccupiedAverage",
        "StateTreeComplexity",
        "BoardCoverageFull",
        "BranchingFactorVariance",
        "BranchingFactorChangeMaxIncrease",
        "DurationMoves",
        "BranchingFactorChangeAverage",
        "BoardSitesOccupiedMedian",
        "BranchingFactorChangeSign",
        'DecisionFactorMedian',
        "BranchingFactorMaximum",
        "DurationActions",
        'NumAdjacentDirections',
        "PieceNumberMedian",
        "MoveDistanceVariance",
        "DecisionFactorMaxIncrease",
        "DurationTurns",
        "PieceNumberMaximum",
        "DecisionFactorChangeAverage",
        "BranchingFactorChangeLineBestFit"]

In [15]:
def get_brute_features2(df, cols):
    dummy = pd.DataFrame()
    
    for col1 in range(len(cols)):
         c1 = cols[col1]
         
         for col2 in range(col1 + 1, len(cols)):
             c2 = cols[col2]
            
             dummy[c1 + "_+_"  + c2] = df[c1] + df[c2]
             dummy[c1 + "_-_"  + c2] = df[c1] - df[c2]
             dummy[c1 + "_/_"  + c2] = df[c1] / df[c2]
             dummy[c1 + "_*_"  + c2] = df[c1] * df[c2]
             dummy[c2 + "_/_"  + c1] = df[c2] / df[c1]
            
    return dummy

In [16]:
def get_win_condition_num(row):
    I = re.IGNORECASE
    if re.search("chess|checkmate", row, I):
        return 1
    
    elif re.search("(block |blocks|blocked)", row, I):
        return 2
    
    elif re.search(
            r"last [a-z]+ to make|last to make\
            |last [a-z]+ [a-z\s]*able to make\
            |[a-z]+ who made last m\
            |last to [a-z]+\
            |m[a-z]+ (the )*last\
            |last [a-z]+ to [a-z]+"
                    , row, re.IGNORECASE):
                        
        return 3
    
    elif re.search(
            r"capture all of the other player\
                |captures the most counters\
                    |[a-z]+ [a-z]* by captur\
                        |capture[\s]*(all\s)*(of\s)*(the) [a-z]*", row, I):
        return 4 
    
    # only one player has piece left
    # bear off
    elif re.search(
            r"(one [a-z]+ [a-z\s]*piece[s]* left)|(bear off)\
                |remove all of[\s]*[a-z]*[\s]*[a-z]* piec", row, I):
        return 5
    
    # reduces the opponent to
    # player who has most|more piece
    # player with more piece
    elif re.search(
            r"reduces [a-z\s]* [a-z]+ to\
                |player who has (most|more){1} piece\
                    |player with more piec", row, I):
        return 6
    
    # cannot make a legal move
    elif re.search(
            r"cannot make [a\s]*legal\
                |(not possible to make [a\s]*legal)", row, re.IGNORECASE):
        return 7 
    
    # selects an edge
    elif re.search(
            r"(selects an edge)\
                |(previously[\-\s]*unselected edge)", row, re.IGNORECASE):
        return 8
    
    elif re.search(
            r"connect all the [a-z]+\
            |connect[\s]*[a-z]* group[s]* of\
            |connect [a-z]+ of \
            |connect all [a-z]+ edge", row, I):
        return 9
    
    # reach opposite side of the board
    elif re.search(
            r"[a-z]* (who has|to) reach (the\s)*opposite\
                |place[s]* all of their [a-z]* in the opponen", row, re.IGNORECASE):

        return 10
    elif re.search("sowing", row, I):
        return 11
    
    elif re.search("Go ", row, I):
        return 12 
    
    elif re.search("shogi", row, I):
        return 13 
    elif re.search("pie ", row, I):
        return 14 
    elif re.search("murray", row, I):
        return 15 
    elif re.search("halma", row, I):
        return 16 
    elif re.search("draughts", row, I):
        return 17 
    elif re.search("amazon", row, I):
        return 18 
    elif re.search("breakthrough", row, I):
        return 19 
    elif re.search("huff", row, I):
        return 20 
    elif re.search("xiang", row, I):
        return 21 
    elif re.search("haretavl", row, I):
        return 22 
    elif re.search("miliaire", row, I):
        return 23 
    elif re.search("soppi", row, I):
        return 24 
    elif re.search("eyed", row, I):
        return 25 
    elif re.search("ko", row, I):
        return 26 
    elif re.search("zaslavsky", row, I):
        return 27 
    elif re.search("seega", row, I):
        return 28 
    elif re.search("kendall", row, I):
        return 29 
    elif re.search("morris", row, I):
        return 30 
    elif re.search("linnaeus", row, I):
        return 31 
    elif re.search("carnarvon", row, I):
        return 32 
    elif re.search("carter", row, I):
        return 33
    elif re.search("capture", row, I):
        return 34
    elif re.search("place", row, I):
        return 35
    elif re.search("reduce", row, I):
        return 36
    
    else: return 100

In [17]:
import regex as re

def get_win_condition_v2(row):
    I = re.IGNORECASE
    r = row[-60:]
    
    if re.search("threatened", row, I):
        return 1
    elif re.search("no piece", r, I):
        return 2
    elif re.search("count piece", r, I):
        return 3
    elif re.search("count move", r, I):
        return 4
    elif re.search("count site", r, I):
        return 5
    elif re.search("count at", r, I):
        return 6
    elif re.search("no moves", r, I):
        return 7 
    elif re.search("movelimit|turnlimit", r, I):
        return 8
    elif re.search("is line", r, I):
        return 9
    elif re.search("sites around", r, I):
        return 10
    elif re.search("sites occupied", r, I):
        return 11 
    elif re.search("where \"", r, I):
        return 12
    elif re.search("triggered", r, I):
        return 13
    elif re.search("is empty", r, I):
        return 14
    elif re.search("mover win", r, I):
        return 15
    elif re.search("group at", r, I):
        return 16
    elif re.search("finalpoint|mapentry", r, I):
        return 17
    elif re.search("sites side", r, I):
        return 18
    else: return 0
    


In [18]:
def get_rule_df(rules):
    result = {}
    nn = []
    lt = []
    ltp = []
    lnum = []
    lnump = []
    lC = []
    lCp = []
    ld = []
    ldp = []
    lcc = []
    lccp = []
    lb = []
    lbp = []
    
    for rule in rules:
        n =  len(rule)
        nn.append(n)
        
        # only text
        t = n - len(re.sub("\W|\d", "", rule))
        tp = t/n
        
        lt.append(t)
        ltp.append(tp)
        
        # only num
        num = n - len(re.sub("\d", "", rule))
        nump = num/n
        
        lnum.append(num)
        lnump.append(nump)
        
        # capital case
        C = n - len(re.sub("[A-Z]", "", rule))
        Cp = C/n
        
        lC.append(C)
        lCp.append(Cp)
        
        # double quotes
        d = n - len(re.sub('\"', "", rule)) 
        dp = d/n
        
        ld.append(d)
        ldp.append(dp)
        
        # curly columns 
        cc = n - len(re.sub("\{", "", rule)) 
        ccp = cc/n
        
        lcc.append(cc)
        lccp.append(ccp)
        
        # brackets 
        b = n - len(re.sub("\(", "", rule))
        bp = b/n
        
        lb.append(b)
        lbp.append(bp)
        
    result["len"] = nn 
    result["text"] = lt
    result["text_p"] = ltp
    result["num"] = lnum
    result["num_p"] = lnump
    result["cap"] = lC
    result["cap_p"] = lCp
    result["dq"] = ld
    result["dq_p"] = ldp
    result["cc"] = lcc
    result["cc_p"] = lccp
    result["brackets"] = lb
    result["brackets_p"] = lbp
    
    return pd.DataFrame(result)

In [19]:
def split_cols(df):
    global split_enc
    
    splitted = df["agent1"].reset_index(drop=True).str.split("-", expand=True)
    splitted2 = df["agent2"].reset_index(drop=True).str.split("-", expand=True)
    
    splitted = splitted[[1,2,3,4]].copy()
    splitted2 = splitted2[[1,2,3,4]].copy()
    
    agent1_grp_enc = split_enc.transform(splitted[[1,2,3]])
    agent2_grp_enc = split_enc.transform(splitted2[[1,2,3]])
    
    splitted[4] = splitted[4].astype(bool)
    splitted2[4] = splitted2[4].astype(bool)
    
    agent1_grp_enc = pd.DataFrame(agent1_grp_enc, columns = [f"agent1_grp{idx}" for idx in range(1,4)])
    agent2_grp_enc = pd.DataFrame(agent2_grp_enc, columns = [f"agent2_grp{idx}" for idx in range(1,4)])
    
    agent1_grp_enc["agent1_grp4"] = splitted[4]
    agent2_grp_enc["agent2_grp4"] = splitted2[4]
    
    return agent1_grp_enc, agent2_grp_enc

def get_data(test):
    global cols, obj, objs_enc, split_enc, vect1, vect2, features, patterns, board_enc, tfidf_vect
    global features1 
    
    to_enc = ["GameRulesetName", "agent1", "agent2"]
    df = test.to_pandas()
    
    bf_df = get_brute_features(df, features)

    agent1_grp_enc, agent2_grp_enc = split_cols(df)
    pattern_df = get_pattern_df(df, patterns)

    enc = objs_enc.transform(np.array(df[to_enc]).reshape(-1,3))
    v1 = vect1.transform([i for i in df["EnglishRules"]]).toarray()
    v2 = vect2.transform([i for i in df["EnglishRules"]]).toarray()
    
    boards = get_Boards_feat(df)
    
    boards_df = board_enc.fit_transform(boards.to_numpy().reshape(-1,1))
    boards_df = pd.DataFrame(boards_df, columns = ["Board_Enc"])
    
    #lud_tfidf = tfidf_vect.transform([i for i in df["LudRules"]])

    #lud_tfidf = pd.DataFrame(
     #   lud_tfidf.toarray(), columns = [
    #    f"tfidf_{i.replace(' ', '_')}"for i in tfidf_vect.get_feature_names_out()
    #    ])
    
    #grps = df["agent1"].str.split("-", expand=True)[[1,2,3]]
    #grps = grps[1] + grps[2] + grps[3]
    
    #agg_df = get_agg_features(df, grps, aggs, cols, agg_di)

    X = df.drop(columns = obj, axis = 1)
    X = X.join(agent1_grp_enc)
    X = X.join(agent2_grp_enc)
    X = X.join(pd.DataFrame(v1, columns = [f"tf_{idx}" for idx in range(v1.shape[-1])]))
    X = X.join(pd.DataFrame(v2, columns = [f"tf2_{idx}" for idx in range(v2.shape[-1])]))
    X = X.join(pd.DataFrame(enc.reshape(-1,3), columns = to_enc))
    X = X.join(bf_df)
    X = X.join(pattern_df)
    X = X.join(boards_df)
    X["win_condition"] = df["EnglishRules"].apply(get_win_condition_num)
    #X = X.join(agg_df)
    #X = X.join(lud_tfidf)


    rules = df.LudRules.unique()

    rule_df = get_rule_df(rules)
    rule_dict = {k:rule_df.iloc[v] for v, k in enumerate(rules)}
    
    rule_df = []
    for rule in df.LudRules:
        rule_df.append(rule_dict[rule])
    
    rule_df = pd.DataFrame(rule_df)   

    X = X.join(rule_df.reset_index(drop = True))

    #win2 = df["LudRules"].apply(get_win_condition_v2)
    #X["WIN_CON2"] = win2

    #X.drop(to_drop, axis = 1, inplace = True)
    
    return X, df

In [20]:
def predict_(test):
    global features, models, models2, obj, objs_enc, split_enc, cols
    global vect1, vect2, splits, patterns, board_enc#, tfidf_vect#, aggs, cols, agg_di
    global features1, models3

    
    X, df = get_data(test)
    XX = X.copy()
    #x = X[features1]
    #X = X[features]

    #X = X.fillna(0)
    #X = X.replace(-np.inf, -10000)
    #X = X.replace(np.inf, 10000)
    
    preds1 = np.zeros((splits,len(X)))
    preds2 = np.zeros((splits,len(X)))
    #preds3 = np.zeros((splits,len(X)))
    #preds3 = np.zeros((splits,len(X)))
    #preds4 = np.zeros((4,len(X)))
    #preds5 = np.zeros((4,len(X)))
    #preds6 = np.zeros((splits,len(X)))

    # CAT MODEL
    for idx,model in enumerate(models2):
        preds1[idx] = model.predict(X[features1]).clip(-1,1)

    preds1 = np.mean(preds1, axis = 0)
    print(preds1)

    # for LGB Model
    XX["cat_preds"] = preds1
    XX = XX[features].copy()

    for idx,model in enumerate(models):
        preds2[idx] = model.predict(XX).clip(-1,1)

    preds2 = np.mean(preds2, axis = 0)

    print(preds2) 

    # CAT V2 MODEL
#    X["cat_preds"] = preds1 
#    X["lgb_preds"] = preds2
    
#    X = X[features1].copy()
    
#    for idx,model in enumerate(models3):
#        preds3[idx] = model.predict(X).clip(-1,1)
#    for idx,model in enumerate(models3):
#        preds3[idx] = model.predict(x).clip(-1,1)
#
#    for idx,model in enumerate(models4):
#       preds4[idx] = model.predict(X).clip(-1,1)
#
#    for idx,model in enumerate(models5):
#       preds5[idx] = model.predict(X).clip(-1,1)
#
#    for idx,model in enumerate(models6):
#       preds6[idx] = model.predict(XX).clip(-1,1)
    

    #preds1 = np.mean(preds1, axis = 0)
#    preds = np.mean(preds3, axis = 0)
#    preds3 = np.mean(preds3, axis = 0)
#    preds4 = np.mean(preds4, axis = 0)
#    preds5 = np.mean(preds5, axis = 0)
#    preds6 = np.mean(preds6, axis = 0)

#    print(preds1, preds2, preds3, preds4, preds5, preds6)
#    print()
   
    print(len(features), len(features1))
#    preds = (
#        (preds1 * .2) +
#        (preds2 * .35) +
#        (preds3 * .2) + 
#        (preds4 * .025) + 
#        (preds5 * .025) + 
#        (preds6 * .2) 
#    )
    print(preds2)
    return preds2

In [21]:
import os
import kaggle_evaluation.mcts_inference_server
import numpy as np


def predict(test, submission):
    global features, models, models2, obj, objs_enc, split_enc, vect1, vect2, nums, splits, board_enc
    global features1, models3
    
    preds = predict_(test)
    return submission.with_columns(pl.Series('utility_agent1', preds))


inference_server = kaggle_evaluation.mcts_inference_server.MCTSInferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway(
        (
            '/kaggle/input/um-game-playing-strength-of-mcts-variants/test.csv',
            '/kaggle/input/um-game-playing-strength-of-mcts-variants/sample_submission.csv'
        )
    )

TBB Warning: The number of workers is currently limited to 0. The request for 3 workers is ignored. Further requests for more workers will be silently ignored until the limit changes.



[-0.29151524  0.17339536  0.12171333]
[-0.27684262  0.19240186  0.10806213]
478 434
[-0.27684262  0.19240186  0.10806213]


In [22]:
# [ 0.11445595 -0.28195612  0.15925361]